# 🏠 Arquitetura Medallion - Visão Geral

Este notebook executa o pipeline real do projeto nas 3 camadas, com **data quality** entre elas:

1. **Bronze**: ingestão dos CSVs brutos (`data/`) para tabelas Delta + check
2. **Silver**: limpeza, tipagem e normalização + check
3. **Gold**: agregações de negócio + check

Stack: Spark + Delta Lake + MinIO (**padrão ligado** — para warehouse local use `USE_MINIO=0`).

> **Dados faltando?** Se `data/` estiver vazio (repo clonado sem CSVs): `python -m scripts.generate_data`
>
> **Mesma sequência no Airflow**: DAG `medallion_pipeline` com as tasks `bronze >> check >> silver >> check >> gold >> check` — veja a seção final.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.session import get_spark
from src.ingestion.Bronze import run as run_bronze
from src.processing.Silver import run as run_silver
from src.serving.Gold import run as run_gold
from src.dq import checks as dq

spark = get_spark("MedallionOverview")
print("Spark Session criada com sucesso!")

11:30:53 INFO [src.session] Bucket 'lake' já existe no MinIO
11:30:53 INFO [src.session] Storage: MinIO (s3a://lake/warehouse)
26/09/23 11:30:56 WARN Utils: Your hostname, REX resolves to a loopback address: 127.0.1.1; using 192.168.15.5 instead (on interface wlo1)
26/09/23 11:30:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/julio-cesar/Documents/Data-Projects/08-data-lakehouse-medallion/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/julio-cesar/.ivy2/cache
The jars for the packages stored in: /home/julio-cesar/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-305a6c76-a8ee-4205-baea-a2173006d478;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 634ms :: artifacts dl 21ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.a

Spark Session criada com sucesso!


## 1. Camada BRONZE — Ingestão de Dados Brutos

In [2]:
run_bronze(spark)
spark.table("bronze.orders").show(5, truncate=False)

26/09/23 11:31:08 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/09/23 11:31:13 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/09/23 11:31:13 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/09/23 11:31:18 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/09/23 11:31:18 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore julio-cesar@127.0.1.1
26/09/23 11:31:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/09/23 11:31:37 ERROR HiveAlterHandler: Failed to alter table bronze.payments 
26/09/23 11:31:37 WARN HiveExternalCatalog: Could not alter schema of t

+--------+-----------+----------+---------+------------+--------------------------+
|order_id|customer_id|order_date|status   |total_amount|_ingested_at              |
+--------+-----------+----------+---------+------------+--------------------------+
|O000001 |C00442     |2025-01-22|delivered|1981.3      |2026-09-23 11:31:44.013519|
|O000002 |C00354     |2025-07-05|delivered|881.79      |2026-09-23 11:31:44.013519|
|O000003 |C00446     |2025-06-08|shipped  |11074.68    |2026-09-23 11:31:44.013519|
|O000004 |C00373     |2025-11-27|shipped  |976.26      |2026-09-23 11:31:44.013519|
|O000005 |C00387     |2025-10-16|shipped  |1664.67     |2026-09-23 11:31:44.013519|
+--------+-----------+----------+---------+------------+--------------------------+
only showing top 5 rows



### ✔ Data Quality — Bronze
Sanidade estrutural: tabelas existem, não vazias, `_ingested_at` presente.

In [3]:
report = dq.run_bronze(spark)
report.log_summary()
assert report.ok, f"{len(report.errors)} erro(s) de DQ na Bronze"

11:32:16 INFO [src.dq.checks] DQ layer=bronze checks=24 passed=24 errors=0 warnings=0


## 2. Camada SILVER — Limpeza e Tipagem

In [ ]:
run_silver(spark, preview=True)

### ✔ Data Quality — Silver
PKs únicas, `rating` 1–5, domínio de `status`, referencialidade + **WARNs** de valores negativos (não bloqueiam).

In [ ]:
report = dq.run_silver(spark)
report.log_summary()
assert report.ok, f"{len(report.errors)} erro(s) de DQ na Silver"

## 3. Camada GOLD — Agregações de Negócio

In [ ]:
run_gold(spark, show=True)

### ✔ Data Quality — Gold
Métricas não nulas e `avaliacao_media` entre 1 e 5.

In [ ]:
report = dq.run_gold(spark)
report.log_summary()
assert report.ok, f"{len(report.errors)} erro(s) de DQ na Gold"

## 4. Consultas nas tabelas Gold

In [ ]:
print("=== Vendas por Categoria ===")
spark.table("gold.vendas_por_categoria").show(truncate=False)

print("=== Pedidos por Status ===")
spark.table("gold.pedidos_por_status").show(truncate=False)

In [ ]:
print("=== Top 10 Clientes por Gasto ===")
spark.table("gold.resumo_clientes").limit(10).show(truncate=False)

In [ ]:
print("=== SQL: receita total por status ===")
spark.sql('''
    SELECT status,
           SUM(receita_total) AS receita,
           SUM(total_pedidos) AS pedidos
    FROM gold.pedidos_por_status
    GROUP BY status
    ORDER BY receita DESC
''').show(truncate=False)

## 5. Histórico de Transações Delta

In [ ]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "bronze.orders")
history = delta_table.history()

print("=== Histórico (bronze.orders) ===")
history.select(
    "version",
    "timestamp",
    "operation",
    "numOutputRows",
).show(truncate=False)

## 6. Orquestração com Airflow

Esta mesma sequência (com os checks entre as camadas) roda automaticamente na DAG **`medallion_pipeline`**:

```
bronze_ingest >> check_bronze >> silver_process >> check_silver >> gold_aggregate >> check_gold
```

```bash
podman-compose up -d --build
# UI: http://localhost:8080 (admin / admin)
podman exec medallion_airflow_scheduler airflow dags trigger medallion_pipeline
```

Agendamento: `@daily`, `catchup=False`, `max_active_runs=1` (Derby metastore não aceita escrita concorrente).

In [ ]:
spark.stop()
print("\nSessão Spark encerrada.")